# Fine-tune the REMI Composer model for live jamming (call-and-response)

Takes the **REMI + BPE tokenizer and `ComposerGPT` checkpoint** you already trained
and fine-tunes it into a *jamming* model: the user plays a riff, stops, and the
model answers with a few bars.

**How it works.** The tokenizer and weights are kept exactly as they are; the model
is only taught a turn-taking format. Training pairs are sliced out of ordinary MIDI
files in token space: a prompt of 4-8 bars, then a response of 2-4 bars, joined by
one new special token `<RESP>`. Loss is computed **only on the response tokens**, so
the model learns to answer, not to memorise prompts.

**Pipeline:** `MIDI -> (transpose) -> REMI/BPE tokens -> cut on Bar tokens ->
(prompt, response) pairs -> [prompt] <RESP> [response] <EOS> -> fine-tune -> sample`

---

### What is different from the original TSD notebook

The original was written for a miditok **TSD** tokenizer and a HuggingFace
`AutoModelForCausalLM`. The actual assets are neither, so three things had to change:

| | original | here |
|---|---|---|
| bar boundaries | accumulate `TimeShift_b.p.r` values | count `Bar` tokens — REMI has them, TSD does not |
| BPE | assumed raw ids | every id is expanded once at startup, because a single BPE merge can *contain* a `Bar` |
| transposition | remap `Pitch_X` ids | shift pitches on the MIDI and re-tokenize — id remapping is meaningless under BPE |
| model | `AutoModelForCausalLM.from_pretrained` | `ComposerGPT` rebuilt from the checkpoint's weight shapes |
| context | `MAX_SEQ_LEN` 1024 | **511** — `pos_emb` has 1024 rows but only 0-510 were ever trained |

The `<RESP>` token grows the vocab from 10000 to **10001**. Your inference notebook
asserts `vocab_size == len(tokenizer)`; for a jam checkpoint that assert has to become
`vocab_size == len(tokenizer) + 1`, and `RESP_ID = len(tokenizer)`.

In [1]:
# miditok >= 3.0 uses the symusic backend
%pip install -q "miditok>=3.0" symusic torch transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 42.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

`RUN` is the only thing to change between the three fine-tunes. Set it, then run the
notebook top to bottom (or hit **Save Version** to commit that run).

* `"smoke"` — 50 anime files, 1 epoch, no augmentation. Proves the whole path end to
  end in a few minutes. **Run this first.**
* `"anime"` / `"classical"` — the two small corpora, with +/-3 semitone augmentation
  (each file is tokenized 7 times, which is the only honest way to augment under BPE).
* `"lakh"` — 17k files, no augmentation; there is already plenty of data.

`STRIDE_BARS` controls how densely each file is sliced (lower = more, overlapping
samples). `PROMPT_BARS` / `RESPONSE_BARS` are sampled per example so the model
generalises across phrase lengths.

In [2]:
RUN = "classical"          # <<< "smoke" | "anime" | "classical" | "lakh"

ASSETS = "/kaggle/input/datasets/perryplay/remigptbig"
DATA   = "/kaggle/input/datasets"

FULL_TRANSPOSE = [-3, -2, -1, 0, 1, 2, 3]

PRESETS = {
    "smoke":     {"MIDI_DATA_DIR": f"{DATA}/programgeek01/anime-music-midi/data/anime",
                  "OUTPUT_DIR": "/kaggle/working/smoke",
                  "MAX_FILES": 50, "EPOCHS": 1, "TRANSPOSITIONS": [0]},
    "anime":     {"MIDI_DATA_DIR": f"{DATA}/programgeek01/anime-music-midi/data/anime",
                  "OUTPUT_DIR": "/kaggle/working/jam_anime",
                  "TRANSPOSITIONS": FULL_TRANSPOSE},
    "classical": {"MIDI_DATA_DIR": f"{DATA}/soumikrakshit/classical-music-midi",
                  "OUTPUT_DIR": "/kaggle/working/jam_classical",
                  "TRANSPOSITIONS": FULL_TRANSPOSE},
    "lakh":      {"MIDI_DATA_DIR": f"{DATA}/imsparsh/lakh-midi-clean",
                  "OUTPUT_DIR": "/kaggle/working/jam_lakh",
                  "TRANSPOSITIONS": [0]},
}

CONFIG = {
    "TOKENIZER": f"{ASSETS}/Compose_REMI.json",
    "CKPT":      f"{ASSETS}/checkpoint_best.pt",

    # ---- musical slicing ----
    "PROMPT_BARS":         (4, 8),    # sampled uniformly per example
    "RESPONSE_BARS":       (2, 4),
    "STRIDE_BARS":         4,         # step between slice windows within a file
    "MIN_PROMPT_TOKENS":   24,        # skip near-empty bars
    "MIN_RESPONSE_TOKENS": 12,
    "TRANSPOSITIONS":      [0],

    # ---- model / training ----
    "MAX_SEQ_LEN":   511,             # pos_emb 511-1023 were never trained
    "DROPOUT":       0.1,             # matches the checkpoint's config
    "BATCH_SIZE":    8,
    "GRAD_ACCUM":    4,               # effective batch = BATCH_SIZE * GRAD_ACCUM
    "EPOCHS":        3,
    "LR":            3e-5,
    "WARMUP_FRAC":   0.05,
    "WEIGHT_DECAY":  0.01,
    "VAL_FRACTION":  0.05,
    "MAX_FILES":     None,
    "SEED":          42,
}

assert RUN in PRESETS, f"RUN must be one of {sorted(PRESETS)}"
CONFIG.update(PRESETS[RUN])
CONFIG["RUN_NAME"] = RUN

import os, json, math, random, time
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

for k in ("TOKENIZER", "CKPT", "MIDI_DATA_DIR"):
    assert Path(CONFIG[k]).exists(), f"{k} not found: {CONFIG[k]}"

OUT = Path(CONFIG["OUTPUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
random.seed(CONFIG["SEED"]); torch.manual_seed(CONFIG["SEED"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"run={RUN} | device={device} | data={CONFIG['MIDI_DATA_DIR']}")
print(f"out={OUT} | transpositions={CONFIG['TRANSPOSITIONS']}")

run=classical | device=cuda | data=/kaggle/input/datasets/soumikrakshit/classical-music-midi
out=/kaggle/working/jam_classical | transpositions=[-3, -2, -1, 0, 1, 2, 3]


## 2. Tokenizer, and the BPE bar table

We do **not** touch the miditok vocabulary. The `<RESP>` separator lives purely on the
model side: its id is `len(tokenizer)`, one past the existing vocab.

The one piece of real work here is `BARS`. Under BPE a single id can expand to several
base tokens, and one of them may be a `Bar`. So each id is decoded once at startup and
we record how many `Bar` tokens it contains. That table is what lets us cut prompts and
responses on bar lines without touching MIDI. It takes a few seconds to build.

In [3]:
from miditok import REMI, TokSequence

tokenizer  = REMI(params=CONFIG["TOKENIZER"])
VOCAB      = len(tokenizer)
base_vocab = tokenizer.vocab if isinstance(tokenizer.vocab, dict) else \
             {t: i for i, t in enumerate(tokenizer.vocab)}
IS_BPE     = VOCAB > len(base_vocab)
print(f"REMI | vocab {VOCAB} | base {len(base_vocab)} | bpe {IS_BPE}")

def tok_id(name):
    try:
        return tokenizer[name]
    except Exception:
        return None

PAD_ID, EOS_ID, BOS_ID = tok_id("PAD_None"), tok_id("EOS_None"), tok_id("BOS_None")
if PAD_ID is None:
    PAD_ID = 0
    print("WARNING: no PAD_None in vocab, falling back to id 0 for padding")
print("PAD:", PAD_ID, "| EOS:", EOS_ID, "| BOS:", BOS_ID)

assert IS_BPE is False or hasattr(tokenizer, "decode_token_ids"), \
    "this miditok build cannot expand BPE ids; pin a 3.x release with decode_token_ids"

_bar_base = {i for t, i in base_vocab.items() if t.startswith("Bar")}

BARS = [0] * VOCAB
for i in range(VOCAB):
    if not IS_BPE:
        BARS[i] = 1 if i in _bar_base else 0
        continue
    try:
        s = TokSequence(ids=[i], are_ids_encoded=True)
        tokenizer.decode_token_ids(s)
        BARS[i] = sum(1 for t in (s.tokens or [])
                      if isinstance(t, str) and t.startswith("Bar"))
    except Exception:
        BARS[i] = 0

print(f"{sum(1 for b in BARS if b)} of {VOCAB} ids contain a Bar token")
assert any(BARS), "no Bar tokens found -- slicing would produce zero pairs"

REMI | vocab 10000 | base 395 | bpe True
PAD: 0 | EOS: 2 | BOS: 1
461 of 10000 ids contain a Bar token


## 3. Model

`ComposerGPT` is rebuilt exactly as in your training notebook, then the architecture is
read back **from the weight shapes rather than the stored config**, because a hardcoded
config dict can drift from the model that was actually built. Shapes cannot lie.

Growing the vocab by one row is a single operation here: `head.weight` is tied to
`tok_emb.weight`, so replacing the embedding and re-tying covers both matrices. The new
row starts at the mean of the existing embeddings plus small noise.

In [4]:
class _Block(nn.Module):
    def __init__(self, d, h, p):
        super().__init__()
        self.n_heads, self.dropout = h, p
        self.norm1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.attn_out = nn.Linear(d, d, bias=False)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(),
                                 nn.Linear(4 * d, d), nn.Dropout(p))

    def forward(self, x):
        B, L, D = x.shape
        h = self.norm1(x)
        q, k, v = self.qkv(h).chunk(3, dim=-1)
        q = q.view(B, L, self.n_heads, -1).transpose(1, 2)
        k = k.view(B, L, self.n_heads, -1).transpose(1, 2)
        v = v.view(B, L, self.n_heads, -1).transpose(1, 2)
        a = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0)
        x = x + self.attn_out(a.transpose(1, 2).reshape(B, L, D))
        return x + self.mlp(self.norm2(x))


class ComposerGPT(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_layers=8, n_heads=8,
                 max_seq_len=1024, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(_Block(d_model, n_heads, dropout)
                                    for _ in range(n_layers))
        self.norm_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        for b in self.blocks:
            h = b(h)
        return self.head(self.norm_f(h))


ckpt = torch.load(CONFIG["CKPT"], map_location="cpu", weights_only=False)
sd = ckpt.get("model", ckpt)
sd = {k[7:] if k.startswith("module.") else k: v for k, v in sd.items()}   # DataParallel

ck_vocab, D_MODEL = sd["tok_emb.weight"].shape
MAX_POS  = sd["pos_emb.weight"].shape[0]
N_LAYERS = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
N_HEADS  = (ckpt.get("config") or {}).get("n_heads", 8)
assert ck_vocab == VOCAB, f"checkpoint vocab {ck_vocab} != tokenizer {VOCAB}"

model = ComposerGPT(ck_vocab, D_MODEL, N_LAYERS, N_HEADS,
                    max_seq_len=MAX_POS, dropout=CONFIG["DROPOUT"])
model.load_state_dict(sd, strict=True)

# ---- grow the tied embedding + output matrix by one row for <RESP> ----
RESP_ID = ck_vocab
_old = model.tok_emb.weight.data
_emb = nn.Embedding(RESP_ID + 1, D_MODEL)
with torch.no_grad():
    _emb.weight.data[:RESP_ID] = _old
    _emb.weight.data[RESP_ID]  = _old.mean(dim=0) + 0.02 * torch.randn(D_MODEL)
model.tok_emb = _emb
model.head = nn.Linear(D_MODEL, RESP_ID + 1, bias=False)
model.head.weight = model.tok_emb.weight
NEW_VOCAB = RESP_ID + 1
model.to(device)

MAX_CTX = min(CONFIG["MAX_SEQ_LEN"], MAX_POS)
print(f"d_model {D_MODEL} | layers {N_LAYERS} | heads {N_HEADS} | pos rows {MAX_POS}")
print(f"RESP_ID {RESP_ID} | vocab {VOCAB} -> {NEW_VOCAB} | context capped at {MAX_CTX}")
print(f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M | step "
      f"{ckpt.get('global_step')} | prev val {ckpt.get('best_val_loss')}")

d_model 512 | layers 8 | heads 8 | pos rows 1024
RESP_ID 10000 | vocab 10000 -> 10001 | context capped at 511
params 30.8M | step 7000 | prev val 1.5446199050574854


## 4. Build (prompt, response) pairs from MIDI

**Bar boundaries.** `BARS[id]` tells us which token indices open a bar, so prompts and
responses are cut on the grid without touching MIDI. Under BPE the cut lands on the id
that *contains* the `Bar` — the same convention `last_n_bars` uses at inference, so
training prompts look like serving prompts.

**Loss masking.** Each example is `prompt + <RESP> + response (+ EOS)`. Labels are
`-100` over the prompt and the `<RESP>` position, so gradient flows only through
response prediction.

**Transposition** shifts pitches on the `symusic` Score and re-tokenizes. A shift is
skipped when it would push any note outside the tokenizer's pitch range, so notes are
never silently clipped. Drum tracks are left alone.

In [5]:
from symusic import Score
from tqdm.auto import tqdm

def pitch_bounds(tk):
    pr = getattr(getattr(tk, "config", None), "pitch_range", None)
    if pr is None:
        return 0, 127
    if isinstance(pr, range):
        return pr.start, pr.stop - 1
    try:
        return int(pr[0]), int(pr[1]) - 1
    except Exception:
        return 0, 127

PITCH_LO, PITCH_HI = pitch_bounds(tokenizer)
print("pitch range:", PITCH_LO, "-", PITCH_HI)


def pitched_extent(score):
    lo, hi = 128, -1
    for tr in score.tracks:
        if getattr(tr, "is_drum", False):
            continue
        for n in tr.notes:
            lo = min(lo, n.pitch)
            hi = max(hi, n.pitch)
    return None if hi < 0 else (lo, hi)


def transpose_score(score, semis):
    if semis == 0:
        return score
    try:
        s = score.copy(deep=True)
    except TypeError:
        s = score.copy()
    for tr in s.tracks:
        if getattr(tr, "is_drum", False):
            continue
        try:
            tr.shift_pitch(semis, inplace=True)
            continue
        except Exception:
            pass
        for n in tr.notes:
            n.pitch = n.pitch + semis
    return s


def make_pairs_from_ids(ids, rng):
    bounds = [i for i, t in enumerate(ids) if BARS[t]]
    if len(bounds) < 3:
        return []
    n_bars = len(bounds) - 1
    pairs, b = [], 0
    while b < n_bars:
        p = rng.randint(*CONFIG["PROMPT_BARS"])
        r = rng.randint(*CONFIG["RESPONSE_BARS"])
        if b + p + r > n_bars:
            break
        pr = ids[bounds[b]: bounds[b + p]]
        rs = ids[bounds[b + p]: bounds[b + p + r]]
        if (len(pr) >= CONFIG["MIN_PROMPT_TOKENS"]
                and len(rs) >= CONFIG["MIN_RESPONSE_TOKENS"]):
            pairs.append((pr, rs))
        b += CONFIG["STRIDE_BARS"]
    return pairs


root = Path(CONFIG["MIDI_DATA_DIR"])
midi_files = sorted(list(root.rglob("*.mid")) + list(root.rglob("*.midi")))
if CONFIG["MAX_FILES"]:
    midi_files = midi_files[: CONFIG["MAX_FILES"]]
print(len(midi_files), "MIDI files")
assert midi_files, f"no MIDI under {root}"

rng = random.Random(CONFIG["SEED"])
all_pairs, skipped = [], 0

for f in tqdm(midi_files, desc="tokenize+slice"):
    try:
        score = Score(str(f))
    except Exception:
        skipped += 1
        continue
    extent = pitched_extent(score)
    if extent is None:
        skipped += 1
        continue
    pmin, pmax = extent
    for semis in CONFIG["TRANSPOSITIONS"]:
        if pmin + semis < PITCH_LO or pmax + semis > PITCH_HI:
            continue                       # would clip notes -- skip this shift
        try:
            out = tokenizer(transpose_score(score, semis))
        except Exception:
            continue
        for seq in (out if isinstance(out, list) else [out]):
            ids = [int(i) for i in (getattr(seq, "ids", None) or [])]
            if len(ids) < CONFIG["MIN_PROMPT_TOKENS"] + CONFIG["MIN_RESPONSE_TOKENS"]:
                continue
            all_pairs.extend(make_pairs_from_ids(ids, rng))

random.shuffle(all_pairs)
print(len(all_pairs), "training pairs |", skipped, "files skipped")
assert all_pairs, "no pairs produced -- check MIDI_DATA_DIR / MIN_*_TOKENS / BARS"

pitch range: 21 - 108
292 MIDI files


tokenize+slice:   0%|          | 0/292 [00:00<?, ?it/s]

148533 training pairs | 0 files skipped


## 5. Dataset with loss masking

Right padding is safe without an attention mask: attention is causal, so a real token
never attends to a pad, and pad positions carry `-100` labels so they contribute no loss.

In [6]:
from torch.utils.data import Dataset, DataLoader

def build_example(pr, rs):
    tail   = [EOS_ID] if EOS_ID is not None else []
    ids    = list(pr) + [RESP_ID] + list(rs) + tail
    labels = [-100] * (len(pr) + 1) + list(rs) + tail
    if len(ids) > MAX_CTX:
        cut = len(ids) - MAX_CTX
        if cut >= len(pr):                 # response alone longer than context: drop
            return None
        ids, labels = ids[cut:], labels[cut:]   # trim prompt from the left
    return ids, labels


examples = [e for e in (build_example(p, r) for p, r in all_pairs) if e is not None]
n_val = max(1, int(len(examples) * CONFIG["VAL_FRACTION"]))
val_ex, train_ex = examples[:n_val], examples[n_val:]
print(f"{len(train_ex)} train | {len(val_ex)} val | avg "
      f"{sum(len(i) for i, _ in examples)/len(examples):.0f} tokens")


class JamDataset(Dataset):
    def __init__(self, ex): self.ex = ex
    def __len__(self):      return len(self.ex)
    def __getitem__(self, i): return self.ex[i]


def collate(batch):
    n = max(len(i) for i, _ in batch)
    inp = torch.full((len(batch), n), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), n), -100,   dtype=torch.long)
    for r, (i, l) in enumerate(batch):
        inp[r, :len(i)] = torch.tensor(i, dtype=torch.long)
        lab[r, :len(l)] = torch.tensor(l, dtype=torch.long)
    return inp, lab


train_dl = DataLoader(JamDataset(train_ex), batch_size=CONFIG["BATCH_SIZE"],
                      shuffle=True, collate_fn=collate, num_workers=2)
val_dl   = DataLoader(JamDataset(val_ex), batch_size=CONFIG["BATCH_SIZE"],
                      shuffle=False, collate_fn=collate, num_workers=2)

141074 train | 7424 val | avg 202 tokens


## 6. Fine-tune

A conservative recipe on purpose: low LR (`3e-5`) with cosine decay and short warmup,
gradient clipping at 1.0, mixed precision on GPU, and the best checkpoint kept by
validation loss. Since only response tokens carry loss, val loss here directly measures
*how well does it answer*.

Watch the first eval — if val loss barely moves, raise `LR` to `5e-5`; if samples later
sound like the model forgot how to play, lower it to `1e-5` and/or reduce `EPOCHS`.

In [7]:
from transformers import get_cosine_schedule_with_warmup

optim = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"],
                          weight_decay=CONFIG["WEIGHT_DECAY"])
steps_per_epoch = math.ceil(len(train_dl) / CONFIG["GRAD_ACCUM"])
total_steps = max(1, steps_per_epoch * CONFIG["EPOCHS"])
sched = get_cosine_schedule_with_warmup(
    optim, int(total_steps * CONFIG["WARMUP_FRAC"]), total_steps)

use_bf16  = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and not use_bf16))
print(f"{total_steps} optimiser steps | amp {amp_dtype if device=='cuda' else 'off'}")


def loss_fn(logits, labels):
    return F.cross_entropy(
        logits[:, :-1].reshape(-1, logits.shape[-1]).float(),
        labels[:, 1:].reshape(-1), ignore_index=-100)


@torch.no_grad()
def evaluate():
    model.eval()
    tot, n = 0.0, 0
    for inp, lab in val_dl:
        inp, lab = inp.to(device), lab.to(device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device == "cuda"):
            logits = model(inp)
        tot += float(loss_fn(logits, lab)) * inp.shape[0]
        n += inp.shape[0]
    return tot / max(1, n)


CKPT_OUT = OUT / "checkpoint_best.pt"
history, best, gstep = [], float("inf"), 0


def save_checkpoint(epoch, val_loss):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optim.state_dict(),
        "scheduler": sched.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best_val_loss": val_loss,
        "history": history,
        "config": {"d_model": D_MODEL, "n_layers": N_LAYERS, "n_heads": N_HEADS,
                   "dropout": CONFIG["DROPOUT"], "vocab_size": NEW_VOCAB,
                   "seq_len": MAX_CTX},
        "global_step": gstep,
        "resp_id": RESP_ID,
        "dataset": RUN,
    }, CKPT_OUT)


print(f"initial val loss {evaluate():.4f}")

for epoch in range(CONFIG["EPOCHS"]):
    model.train()
    running, seen, t0 = 0.0, 0, time.time()
    optim.zero_grad(set_to_none=True)
    for i, (inp, lab) in enumerate(train_dl):
        inp, lab = inp.to(device), lab.to(device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device == "cuda"):
            logits = model(inp)
        loss = loss_fn(logits, lab)
        scaler.scale(loss / CONFIG["GRAD_ACCUM"]).backward()
        running += float(loss); seen += 1
        if (i + 1) % CONFIG["GRAD_ACCUM"] == 0 or i + 1 == len(train_dl):
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step()
            optim.zero_grad(set_to_none=True)
            gstep += 1
        if seen % 200 == 0:
            print(f"  epoch {epoch} step {i+1}/{len(train_dl)} "
                  f"| loss {running/seen:.4f} | {time.time()-t0:.0f}s", flush=True)
    val = evaluate()
    history.append({"epoch": epoch, "train": running / max(1, seen), "val": val})
    print(f"epoch {epoch}: train {running/max(1,seen):.4f} | val {val:.4f}")
    if val < best:
        best = val
        save_checkpoint(epoch, val)
        print(f"  saved (best) -> {CKPT_OUT}")

13227 optimiser steps | amp torch.bfloat16
initial val loss 3.0822


/tmp/ipykernel_22/933725730.py:69: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  running += float(loss); seen += 1


  epoch 0 step 200/17635 | loss 3.0452 | 59s
  epoch 0 step 400/17635 | loss 2.8056 | 117s
  epoch 0 step 600/17635 | loss 2.6433 | 176s
  epoch 0 step 800/17635 | loss 2.5443 | 236s
  epoch 0 step 1000/17635 | loss 2.4698 | 292s
  epoch 0 step 1200/17635 | loss 2.4137 | 351s
  epoch 0 step 1400/17635 | loss 2.3630 | 407s
  epoch 0 step 1600/17635 | loss 2.3271 | 466s
  epoch 0 step 1800/17635 | loss 2.2932 | 525s
  epoch 0 step 2000/17635 | loss 2.2656 | 583s
  epoch 0 step 2200/17635 | loss 2.2376 | 642s
  epoch 0 step 2400/17635 | loss 2.2143 | 701s
  epoch 0 step 2600/17635 | loss 2.1943 | 760s
  epoch 0 step 2800/17635 | loss 2.1783 | 817s
  epoch 0 step 3000/17635 | loss 2.1643 | 876s
  epoch 0 step 3200/17635 | loss 2.1482 | 935s
  epoch 0 step 3400/17635 | loss 2.1364 | 992s
  epoch 0 step 3600/17635 | loss 2.1255 | 1050s
  epoch 0 step 3800/17635 | loss 2.1145 | 1106s
  epoch 0 step 4000/17635 | loss 2.1029 | 1164s
  epoch 0 step 4200/17635 | loss 2.0945 | 1221s
  epoch 0 step

## 7. Sample, and write the run summary

No KV cache here — `ComposerGPT` does not expose one, so every token re-runs the full
context. That is fine on GPU for a couple of bars. Generation stops once `n_bars` `Bar`
tokens have been emitted (counted through the same BPE-aware `BARS` table), or at
`max_new_tokens`.

`stop="bars"` means the response is bar-aligned and safe to schedule on the beat grid;
`stop="max_tokens"` means it was cut mid-bar.

In [8]:
BANNED = [b for b in (PAD_ID, BOS_ID) if b is not None]

@torch.no_grad()
def generate(prompt_ids, n_bars=2, temperature=1.0, top_p=0.9,
             max_new_tokens=512, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    ids, out, bars, stop = list(prompt_ids)[-MAX_CTX:], [], 0, "max_tokens"
    for _ in range(max_new_tokens):
        ctx = torch.tensor([ids[-MAX_CTX:]], dtype=torch.long, device=device)
        lg = model(ctx)[0, -1].float() / max(temperature, 1e-5)
        for b in BANNED:
            lg[b] = float("-inf")
        probs = torch.softmax(lg, dim=-1)
        sp, si = torch.sort(probs, descending=True)
        keep = int((torch.cumsum(sp, -1) < top_p).sum().item()) + 1
        sp, si = sp[:keep], si[:keep]
        nxt = int(si[torch.multinomial(sp / sp.sum(), 1)])
        bars += BARS[nxt] if nxt < len(BARS) else 0
        if bars > n_bars:
            stop = "bars"; break
        out.append(nxt); ids.append(nxt)
        if bars == n_bars and nxt < len(BARS) and BARS[nxt]:
            stop = "bars"; break
    return out, stop


best_ck = torch.load(CKPT_OUT, map_location="cpu", weights_only=False)
model.load_state_dict(best_ck["model"]); model.to(device)

pr, _ = random.choice(all_pairs)
prompt = list(pr)[-(MAX_CTX - 64):] + [RESP_ID]
gen, stop = generate(prompt, n_bars=2, seed=0)
clean = [i for i in gen if i != RESP_ID]
print(f"sample: {len(gen)} tokens | stop={stop}")

if clean:
    sc = tokenizer.decode([TokSequence(ids=clean, are_ids_encoded=IS_BPE)])
    sc.dump_midi(str(OUT / "sample_response.mid"))
    print("decoded to", sum(len(t.notes) for t in sc.tracks),
          "notes ->", OUT / "sample_response.mid")
else:
    print("empty response -- see the diagnosis cells in your inference notebook")

json.dump({"run": RUN, "best_val_loss": best, "history": history,
           "resp_id": RESP_ID, "vocab_size": NEW_VOCAB, "seq_len": MAX_CTX,
           "n_pairs": len(all_pairs), "n_files": len(midi_files),
           "config": {k: v for k, v in CONFIG.items()}},
          open(OUT / "summary.json", "w"), indent=2, default=str)

print(f"\ndone | run={RUN} | best val {best:.4f} | {CKPT_OUT}")

sample: 5 tokens | stop=bars
decoded to 1 notes -> /kaggle/working/jam_classical/sample_response.mid

done | run=classical | best val 1.4065 | /kaggle/working/jam_classical/checkpoint_best.pt


## Running the other datasets

Change `RUN` in cell 1 and re-run, once per dataset. With **Save Version** each run is
committed separately, so a failure on one does not cost the others:

1. `RUN = "smoke"` — sanity check, minutes
2. `RUN = "anime"` — `/kaggle/working/jam_anime/checkpoint_best.pt`
3. `RUN = "classical"` — `/kaggle/working/jam_classical/checkpoint_best.pt`
4. `RUN = "lakh"` — `/kaggle/working/jam_lakh/checkpoint_best.pt`

### Using a jam checkpoint in your inference notebook

Two edits in *Test the REMI jam model*, cell 3:

```python
RESP_ID = len(tokenizer)                       # 10000
assert vocab_size == RESP_ID + 1               # was: == VOCAB
```

then prime generation with `prompt = last_n_bars(encode(RIFF), 8) + [RESP_ID]` and strip
`RESP_ID` out of the generated ids before decoding.